# Fine-tuning Qwen2.5-1.5B com LoRA para pedidos de delivery em PT-BR

Este notebook treina e avalia o modelo do projeto [safe-order-parser-ptbr](https://github.com/marianamourapy/safe-order-parser-ptbr): um Qwen2.5-1.5B-Instruct ajustado com LoRA que converte mensagens informais de delivery em um pedido estruturado em JSON, com restrições alimentares, alerta de saúde e uma intenção (pedido, fora de escopo, emergência ou ataque).

**Adaptador publicado:** [namxsj/safe-order-parser-ptbr-qwen2.5-1.5b-lora](https://huggingface.co/namxsj/safe-order-parser-ptbr-qwen2.5-1.5b-lora)

## O que o notebook faz
Avalia quatro sistemas nas mesmas 641 frases de teste, congeladas com hash antes de qualquer avaliação:

| Sistema | Emergência detectada | Ataque detectado | Acerto exato (conjunto principal) |
|---|---|---|---|
| Regras escritas à mão | 57% | 50% | 66% |
| Qwen sem ajuste | 0% | 0% | 3% |
| Qwen + 8 exemplos no prompt | 20% | 0% | 20% |
| **Qwen + LoRA** | **97%** | **90%** | **66%** |

Resultados completos, análise de erros e limitações no README do repositório.

## Como executar
1. Adicione o dataset com o projeto: painel direito → **Add Input**.
2. Em **Settings**: Accelerator **GPU T4 x2** e Internet **on**.
3. **Run All**. O treino leva cerca de 40 minutos; o notebook inteiro, cerca de 1 hora.
4. No fim, baixe `resultados.zip` em **Output**.

Também funciona no Google Colab com GPU T4: a primeira célula pede o zip do projeto.

## 0. Ambiente
Remove uma versão do `torchao` pré-instalada no Kaggle que é incompatível com o `peft`.

In [ ]:
!pip uninstall -y -q torchao

## 1. Projeto
Carrega o código e os dados, instala as dependências e roda os testes automatizados do repositório.

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # no Kaggle (T4 x2), usa uma GPU só
import sys, zipfile, glob, subprocess, json, time, gc, random, shutil

NO_KAGGLE = os.path.exists("/kaggle")
BASE = "/kaggle/working" if NO_KAGGLE else "/content"
PROJ = f"{BASE}/proj"

if not os.path.exists(f"{PROJ}/src/order_parser/schema.py"):
    if NO_KAGGLE:
        zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
        pastas = glob.glob("/kaggle/input/**/src/order_parser/schema.py", recursive=True)
        if zips:
            zipfile.ZipFile(zips[0]).extractall(PROJ)
        elif pastas:   # o Kaggle descompacta zips enviados como Dataset
            shutil.copytree(pastas[0].rsplit("/src/", 1)[0], PROJ)
        else:
            raise FileNotFoundError("Adicione o dataset com o projeto: painel direito > Add Input.")
    else:
        locais = glob.glob("/content/*.zip")
        if locais:
            zipfile.ZipFile(locais[0]).extractall(PROJ)
        else:
            from google.colab import files
            up = files.upload()
            zipfile.ZipFile(next(iter(up))).extractall(PROJ)

os.chdir(PROJ)
sys.path.insert(0, f"{PROJ}/src")
os.environ["PYTHONPATH"] = f"{PROJ}/src"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "pydantic", "openpyxl"], check=True)
print(subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], capture_output=True, text=True).stdout[-200:])

## 2. Modelo e funções de geração
Carrega o tokenizador, define a geração em lote (decodificação gulosa, fp16) e a função que salva as previsões e calcula as métricas.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from order_parser.output_format import SISTEMA, mensagens, para_json
from order_parser.schema import Pedido

assert torch.cuda.is_available(), "Ligue a GPU: Ambiente de execução > Alterar tipo > T4"
MODELO = "Qwen/Qwen2.5-1.5B-Instruct"
SEED = 7
random.seed(SEED); torch.manual_seed(SEED)

tok = AutoTokenizer.from_pretrained(MODELO)
tok.padding_side = "left"   # na geração em lote, o preenchimento fica à esquerda

def carregar_base(dtype):
    try:
        return AutoModelForCausalLM.from_pretrained(MODELO, dtype=dtype, device_map="cuda")
    except TypeError:   # versões antigas do transformers
        return AutoModelForCausalLM.from_pretrained(MODELO, torch_dtype=dtype, device_map="cuda")

CONJUNTOS = sorted(glob.glob("data/test/test_*.jsonl"))
TESTE = [json.loads(l) for f in CONJUNTOS for l in open(f, encoding="utf8")]
print(len(TESTE), "frases de teste em", len(CONJUNTOS), "conjuntos")
LAT = {}

@torch.no_grad()
def gerar(model, lista_msgs, max_new_tokens, lote=16):
    prompts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in lista_msgs]
    ordem = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    saidas = [None] * len(prompts); n_in = n_out = 0
    t0 = time.time()
    for k in range(0, len(ordem), lote):
        idx = ordem[k:k + lote]
        enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True).to("cuda")
        with torch.autocast("cuda", dtype=torch.float16):
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
        novos = out[:, enc["input_ids"].shape[1]:]
        for i, seq in zip(idx, novos):
            saidas[i] = tok.decode(seq, skip_special_tokens=True).strip()
            n_out += int((seq != tok.pad_token_id).sum())
        n_in += int(enc["attention_mask"].sum())
    dt = time.time() - t0
    return saidas, {"tokens_entrada": n_in / len(prompts), "tokens_saida": n_out / len(prompts), "seg_por_msg": dt / len(prompts)}

def avaliar(slug, nome, saidas, stats):
    os.makedirs("results", exist_ok=True)
    with open(f"results/preds_{slug}.jsonl", "w", encoding="utf8") as f:
        for ex, s in zip(TESTE, saidas):
            f.write(json.dumps({"id": ex["id"], "saida": s}, ensure_ascii=False) + "\n")
    LAT[slug] = stats
    json.dump(LAT, open("results/latency.json", "w"), indent=1)
    r = subprocess.run([sys.executable, "scripts/evaluate.py", "--preds", f"results/preds_{slug}.jsonl", "--nome", nome, "--slug", slug],
                       capture_output=True, text=True)
    print(r.stdout[:1600], r.stderr[-800:])

## 3. Baseline: Qwen sem ajuste
O modelo base recebe só a instrução do sistema, sem exemplos. Mostra o ponto de partida antes de qualquer adaptação.

In [ ]:
model = carregar_base(torch.float16)
model.eval()
saidas, stats = gerar(model, [mensagens(ex["texto"]) for ex in TESTE], max_new_tokens=256)
print("exemplo:", TESTE[0]["texto"], "->", saidas[0][:200])
avaliar("zero_shot", "Qwen sem ajuste", saidas, stats)

## 4. Baseline: Qwen com 8 exemplos no prompt
A alternativa barata ao fine-tuning. Os 8 exemplos vêm do treino e cobrem cada tipo de caso. Sem esta comparação, não daria para dizer se treinar foi necessário.

In [ ]:
TREINO = [json.loads(l) for l in open("data/train/train.jsonl", encoding="utf8")]
VAL = [json.loads(l) for l in open("data/train/val.jsonl", encoding="utf8")]

def escolher_exemplos(dados):
    r = lambda ex: ex["rotulo"]
    criterios = [
        lambda ex: r(ex)["intent"] == "emergencia",
        lambda ex: r(ex)["intent"] == "ataque",
        lambda ex: r(ex)["intent"] == "fora_escopo",
        lambda ex: r(ex)["intent"] == "pedido" and ex["saida"] == "{}",
        lambda ex: r(ex)["health_alert"] and len(r(ex)["restrictions"]) >= 1 and r(ex)["categories"],
        lambda ex: r(ex)["partial_restrictions"] and r(ex)["people"],
        lambda ex: r(ex)["exclude_categories"] and not r(ex)["categories"],
        lambda ex: r(ex)["max_price"] and r(ex)["free_delivery"] and r(ex)["categories"],
    ]
    return [next(ex for ex in dados if c(ex)) for c in criterios]

EXEMPLOS = escolher_exemplos(TREINO)
for ex in EXEMPLOS: print(ex["texto"], "->", ex["saida"])

def msgs_few_shot(texto):
    m = [{"role": "system", "content": SISTEMA}]
    for ex in EXEMPLOS:
        m += [{"role": "user", "content": ex["texto"]}, {"role": "assistant", "content": ex["saida"]}]
    return m + [{"role": "user", "content": texto}]

saidas, stats = gerar(model, [msgs_few_shot(ex["texto"]) for ex in TESTE], max_new_tokens=256, lote=8)
avaliar("few_shot", "Qwen + 8 exemplos no prompt", saidas, stats)
del model; gc.collect(); torch.cuda.empty_cache()

## 5. Fine-tuning com LoRA
LoRA de rank 16 em todas as projeções de atenção e MLP (1,18% dos parâmetros), 2 épocas, lote efetivo de 16 com gradient checkpointing para caber nos 16 GB da T4. A perda só é calculada nos tokens da resposta.

In [ ]:
import math
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments

LAT = json.load(open("results/latency.json"))   # recupera os tempos das células 3 e 4
TREINO = [json.loads(l) for l in open("data/train/train.jsonl", encoding="utf8")]
VAL = [json.loads(l) for l in open("data/train/val.jsonl", encoding="utf8")]
print("sistemas já avaliados:", list(LAT), "| memória na GPU:", f"{torch.cuda.memory_allocated() / 1e9:.1f} GB")

EPOCAS, LR, RANK = 2, 2e-4, 16
base = carregar_base(torch.float32)
cfg = LoraConfig(r=RANK, lora_alpha=2 * RANK, lora_dropout=0.05, task_type="CAUSAL_LM",
                 target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(base, cfg)
model.print_trainable_parameters()
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model.config.use_cache = False

class Dados(torch.utils.data.Dataset):
    def __init__(self, exemplos):
        self.itens = []
        for ex in exemplos:
            prompt = tok.apply_chat_template(mensagens(ex["texto"]), tokenize=False, add_generation_prompt=True)
            p = tok(prompt, add_special_tokens=False)["input_ids"]
            r = tok(ex["saida"] + "<|im_end|>", add_special_tokens=False)["input_ids"]
            self.itens.append({"input_ids": p + r, "labels": [-100] * len(p) + r})
    def __len__(self): return len(self.itens)
    def __getitem__(self, i): return self.itens[i]

def juntar(lote):
    n = max(len(x["input_ids"]) for x in lote)
    ids = torch.full((len(lote), n), tok.pad_token_id); lab = torch.full((len(lote), n), -100); att = torch.zeros((len(lote), n), dtype=torch.long)
    for i, x in enumerate(lote):
        k = len(x["input_ids"])
        ids[i, :k] = torch.tensor(x["input_ids"]); lab[i, :k] = torch.tensor(x["labels"]); att[i, :k] = 1
    return {"input_ids": ids, "attention_mask": att, "labels": lab}

d_treino, d_val = Dados(TREINO), Dados(VAL)
passos = math.ceil(len(d_treino) / 16) * EPOCAS
args = TrainingArguments(output_dir="/tmp/saida_treino", num_train_epochs=EPOCAS, per_device_train_batch_size=4,
                         per_device_eval_batch_size=8, gradient_accumulation_steps=4, learning_rate=LR,
                         lr_scheduler_type="cosine", warmup_steps=max(1, int(0.05 * passos)), logging_steps=20,
                         eval_strategy="epoch", save_strategy="no", fp16=True, report_to="none", seed=SEED)
print("passos:", passos, "| memória antes do treino:", f"{torch.cuda.memory_allocated() / 1e9:.1f} GB")
trainer = Trainer(model=model, args=args, train_dataset=d_treino, eval_dataset=d_val, data_collator=juntar)
t0 = time.time(); trainer.train(); minutos = (time.time() - t0) / 60
historico = trainer.state.log_history
json.dump({"modelo": MODELO, "epocas": EPOCAS, "lr": LR, "rank": RANK, "minutos_treino": minutos, "seed": SEED, "log": historico},
          open("results/training_log.json", "w"), indent=1)
print(f"treino: {minutos:.1f} min | perda de validação por época:", [round(h["eval_loss"], 4) for h in historico if "eval_loss" in h])

## 6. Avaliação do modelo ajustado
Gera as previsões nas 641 frases de teste e salva o adaptador.

In [ ]:
model.eval()
saidas, stats = gerar(model, [mensagens(ex["texto"]) for ex in TESTE], max_new_tokens=128)
print("exemplo:", TESTE[0]["texto"], "->", saidas[0])
avaliar("lora", "Qwen + LoRA", saidas, stats)
model.save_pretrained(f"{PROJ}/adapter")
tok.save_pretrained(f"{PROJ}/adapter")

## 7. Tabela comparativa
Junta os quatro sistemas numa tabela com intervalos de confiança de 95% e empacota os resultados para download.

In [ ]:
print(subprocess.run([sys.executable, "scripts/compare.py"], capture_output=True, text=True).stdout)
destino = f"{BASE}/resultados.zip"
with zipfile.ZipFile(destino, "w", zipfile.ZIP_DEFLATED) as z:
    for pasta in ["results", "adapter"]:
        for raiz, _, arqs in os.walk(pasta):
            for a in arqs: z.write(os.path.join(raiz, a))
if NO_KAGGLE:
    print("Pronto. Baixe resultados.zip no painel direito: Output > /kaggle/working.")
else:
    from google.colab import files
    files.download(destino)

## 8. Validação pré-registrada em frases novas
A análise de erros sugeriu um sistema híbrido: o modelo classifica a intenção e as regras extraem os campos. Como ele foi desenhado olhando o teste, é avaliado aqui em 78 frases novas, segundo um [pré-registro](https://github.com/marianamourapy/safe-order-parser-ptbr/blob/main/PREREGISTRATION.md) feito antes da rotulagem. Carrega o adaptador publicado; não precisa treinar de novo.

In [ ]:
from peft import PeftModel
from kaggle_secrets import UserSecretsClient
ADAPTER = "namxsj/safe-order-parser-ptbr-qwen2.5-1.5b-lora"
token = UserSecretsClient().get_secret("HF_TOKEN")
NOVO = [json.loads(l) for l in open("data/test/test_fresh_llm.jsonl", encoding="utf8")]
print(len(NOVO), "frases")
model = PeftModel.from_pretrained(carregar_base(torch.float16), ADAPTER, token=token).eval()
saidas, stats = gerar(model, [mensagens(ex["texto"]) for ex in NOVO], max_new_tokens=128)
os.makedirs("results", exist_ok=True)
with open("results/preds_lora_fresh.jsonl", "w", encoding="utf8") as f:
    for ex, s in zip(NOVO, saidas):
        f.write(json.dumps({"id": ex["id"], "saida": s}, ensure_ascii=False) + "\n")
r = subprocess.run([sys.executable, "scripts/evaluate_fresh.py", "results/preds_lora_fresh.jsonl"],
                   capture_output=True, text=True)
print(r.stdout, r.stderr[-1500:])

## Apêndice A. Publicar o adaptador no Hugging Face
Opcional. Requer um token do Hugging Face guardado em **Add-ons → Secrets** com o nome `HF_TOKEN`.

In [ ]:
from kaggle_secrets import UserSecretsClient
REPO = "namxsj/safe-order-parser-ptbr-qwen2.5-1.5b-lora"
token = UserSecretsClient().get_secret("HF_TOKEN")
model.push_to_hub(REPO, token=token, private=True)
tok.push_to_hub(REPO, token=token, private=True)
print("publicado (privado):", REPO)

## Apêndice B. Demo interativa
Opcional. Abre a demo com o modelo ajustado e o filtro sobre um catálogo fictício. O link `gradio.live` fica ativo enquanto a sessão estiver aberta.

In [ ]:
!pip install -q "gradio<6"
import torch
sys.path.insert(0, f"{PROJ}/demo")
import app as demo
from order_parser.output_format import de_json

def interpretar(texto):
    prompt = tok.apply_chat_template(mensagens(texto), tokenize=False, add_generation_prompt=True)
    enc = tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=128, do_sample=False, pad_token_id=tok.pad_token_id)
    bruto = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        return de_json(bruto), "modelo ajustado"
    except Exception:
        return Pedido(), "modelo ajustado (saída inválida, tratada como pedido vazio)"

demo.interpretar = interpretar
demo.app.launch(share=True)